# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

We evaluate two typical findings from SEO Machine Learning research using a rigorous methodology audit:

**Finding 1:** *"Content older than 180 days with a CTR under 2% will inevitably drop in rankings within the next 30 days."*
- **My Methodology Question:** Where does the label come from, and how is the time window structured? If the label is defined strictly by an observed rank drop *after* the 180-day mark, the timeline is clean. However, does the validation design account for seasonal trends or core algorithm updates? To make this claim trustworthy, I would ask if they used a **time-aware split** to prove this wasn't just an artifact of one specific historical Google update.

**Finding 2:** *"Our NLP model classifies search intent shifts and predicts traffic decay with 92% precision."*
- **My Methodology Question:** Does the validation design actually support that 92% claim? If the model was evaluated using a naive random split, it is highly likely that sibling pages from the same large domain ended up in both the train and test sets. Memorizing a specific domain's overall traffic decline is not the same as generalizing intent shifts. I would respectfully ask if a **grouped split** (by client or domain) was used to ensure true out-of-sample skill before trusting the 92% metric.

## 2. My model under an honest split (before/after)

A random split lets a model memorize client-specific baselines, faking its skill. We train the exact same Random Forest twice to prove this: once on a **Naive Random Split**, and once on an **Honest Grouped Split** (`client_id`). The gap between them shows how much memorization was happening.

In [1]:
# 1. Setup environment and load data
import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-seo-ml-pipeline"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ak8x6/flyrank-seo-ml-pipeline", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) in ("notebooks", "work"):
    os.chdir(os.path.join("..", ".."))

df = pd.read_csv("data/raw/content_refresh_anonymized.csv").fillna(0)
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

features = ["impressions_90d", "days_since_last_update", "avg_position", "word_count", "ctr"]

# Metric helper
def precision_at_k(model, X, y, k=50):
    probs = model.predict_proba(X)[:, 1]
    order = np.argsort(-probs)
    return np.asarray(y)[order[:k]].mean()

# --- 2A. NAIVE RANDOM SPLIT (The Illusion) ---
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(
    df[features], df["is_declining_label"], test_size=0.20, random_state=42
)
rf_random = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_random.fit(X_train_rnd, y_train_rnd)
p50_random = precision_at_k(rf_random, X_test_rnd, y_test_rnd, 50)

# --- 2B. HONEST GROUPED SPLIT (The Truth) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
X_train_grp, X_test_grp = df.iloc[train_idx][features], df.iloc[test_idx][features]
y_train_grp, y_test_grp = df.iloc[train_idx]["is_declining_label"], df.iloc[test_idx]["is_declining_label"]

rf_grouped = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_grouped.fit(X_train_grp, y_train_grp)
p50_grouped = precision_at_k(rf_grouped, X_test_grp, y_test_grp, 50)
base_rate = y_test_grp.mean()

print("--- SPLIT DESIGN: BEFORE & AFTER ---")
print(f"Base Rate (Test Set):          {base_rate:.3f}")
print(f"Naive Random Split P@50:       {p50_random:.3f}  <-- Inflated by client memorization")
print(f"Honest Grouped Split P@50:     {p50_grouped:.3f}  <-- True out-of-sample generalization")
print(f"\nGap Analysis: The {p50_random - p50_grouped:.3f} drop in precision proves the random split was leaking client-specific baseline traffic.")

--- SPLIT DESIGN: BEFORE & AFTER ---
Base Rate (Test Set):          0.511
Naive Random Split P@50:       0.900  <-- Inflated by client memorization
Honest Grouped Split P@50:     0.660  <-- True out-of-sample generalization

Gap Analysis: The 0.240 drop in precision proves the random split was leaking client-specific baseline traffic.


## 3. Leakage audit

We run the attack checklist on our feature set to ensure no answers snuck in. We will deliberately inject a leaky feature (derived from the label) to prove our test harness works, watch the score artificially jump to near 1.0, and then remove it.

In [2]:
# The Attack-Your-Own-Model Leakage Test

# 1. Inject a leaky feature (derived directly from the target)
df["LEAKY_trend_score"] = df["is_declining_label"] * 0.95 + np.random.normal(0, 0.05, len(df))
features_leaky = features + ["LEAKY_trend_score"]

X_train_leak, X_test_leak = df.iloc[train_idx][features_leaky], df.iloc[test_idx][features_leaky]

# 2. Train with the leaky feature
rf_leaky = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_leaky.fit(X_train_leak, y_train_grp)
p50_leaky = precision_at_k(rf_leaky, X_test_leak, y_test_grp, 50)

print("--- THE LEAKAGE AUDIT ---")
print(f"Score WITH leaky label-derived feature: P@50 = {p50_leaky:.3f} (Suspiciously perfect!)")
print(f"Score AFTER removing leaky feature:     P@50 = {p50_grouped:.3f} (The honest score)")

print("\n--- LEAKAGE CHECKLIST CONFIRMATION ---")
print("[x] Timeline drawn: All features (impressions, staleness) are knowable before the decline label window.")
print("[x] No label-derived columns in the final feature set (trend_direction/trend_pct strictly excluded).")
print("[x] No product flags / existing scores used as features.")
print("[x] Split grouped by client_id to prevent memorization.")

--- THE LEAKAGE AUDIT ---
Score WITH leaky label-derived feature: P@50 = 1.000 (Suspiciously perfect!)
Score AFTER removing leaky feature:     P@50 = 0.660 (The honest score)

--- LEAKAGE CHECKLIST CONFIRMATION ---
[x] Timeline drawn: All features (impressions, staleness) are knowable before the decline label window.
[x] No label-derived columns in the final feature set (trend_direction/trend_pct strictly excluded).
[x] No product flags / existing scores used as features.
[x] Split grouped by client_id to prevent memorization.


## 4. Claim rewrite

**Original (Dangerous) Claim:**
*"My Random Forest model successfully predicts exactly which pages will lose traffic because they are getting stale."*

**Rewritten (Safe & Honest) Claim:**
*"The Random Forest model provides a directional, decision-support priority score. When evaluated on entirely unseen test clients, high-scoring pages demonstrated a measured, observable probability of subsequent traffic decline, identifying content staleness as a highly correlated feature."*

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.